Find Source and Sink nodes.
Find tranistion nodes where there is only 1 possiblity
See what happens when I restirct to only pia icaos and real icaos that are known to use pia.

In [1]:
from data_engineering.adsb.read_adsb import pia_or_american_ladd
from flights.evaluation.evaluation import pia_or_american_ladd_expr
from pia.graph import create_graph_from_candidates
from pia.resolver import build_candidates
from datetime import date
from data_engineering.flights.flight_type import FlightIndex, flights_df_to_flights, get_flights, pia_or_american_ladd_icao_filter
import polars as pl
import networkx as nx
start_dt = date(2026, 6, 1)
end_dt = date(2026, 7, 1)
df_flights = get_flights(start_dt, end_dt)
df_flights = df_flights.filter(pia_or_american_ladd_icao_filter())
flights = flights_df_to_flights(df_flights)
flight_index = FlightIndex(flights, None)
candidates = build_candidates(flight_index)
graph = create_graph_from_candidates(candidates, flight_index)

Loading airports from /Volumes/T2-SSD/planequery/data/raw/ourairports/airports.csv


In [7]:
first_node = "a1b6e3_2026-06-09_22-35"
last_node = "a1b6e3_2026-06-30_23-20"
paths = nx.all_simple_paths(graph, source=first_node, target=last_node)
paths = list(paths)

In [ ]:
path_nodes = {
    node
    for path in paths
    for node in path
}

reduced_graph = graph.subgraph(path_nodes).copy()


In [10]:
path_flights = [
    flight_index.get_flight_by_id(node_id)
    for node_id in reduced_graph.nodes
]
reduced_flight_index = FlightIndex(
    path_flights,
    flight_index.airport_continuity_tolerance_km,
)

In [11]:
from pia.graph import check_graph_for_overlap
check_graph_for_overlap(reduced_graph, reduced_flight_index)

In [7]:
len(graph.nodes)

166386

In [8]:
source_nodes = [n for n, d in graph.degree() if d == 0]
sink_nodes = [n for n, d in graph.out_degree()] if graph.is_directed() else source_nodes

if graph.is_directed():
    source_nodes = [n for n in graph.nodes if graph.in_degree(n) == 0]
    sink_nodes = [n for n in graph.nodes if graph.out_degree(n) == 0]
else:
    source_nodes = [n for n in graph.nodes if graph.degree(n) == 1]
    sink_nodes = source_nodes

print(f"Number of source nodes: {len(source_nodes)}")
print(f"Number of sink nodes: {len(sink_nodes)}")
print(source_nodes[:10])
print(sink_nodes[:10])

Number of source nodes: 34675
Number of sink nodes: 34357
['a00014_2026-06-01_22-14', 'a0001d_2026-06-01_14-14', 'a0003c_2026-06-01_10-02', 'a0003d_2026-06-01_19-10', 'a00059_2026-06-01_17-56', 'a00062_2026-06-01_17-11', 'a00084_2026-06-01_21-32', 'a00088_2026-06-01_10-28', 'a0008b_2026-06-01_17-28', 'a0009d_2026-06-01_22-06']
['a00059_2026-06-01_19-25', 'a000c9_2026-06-01_21-47', 'a001e0_2026-06-01_18-06', 'a00385_2026-06-01_22-53', 'a004bf_2026-06-01_22-52', 'a004ca_2026-06-01_13-44', 'a00562_2026-06-01_18-17', 'a00639_2026-06-01_18-06', 'a00adc_2026-06-01_18-26', 'a010f7_2026-06-01_15-52']


See If I can detect teleports between american airports

In [13]:
def graph_for_real_icao(
    graph: nx.DiGraph,
    flight_index: FlightIndex,
    real_icao: str,
) -> nx.DiGraph:
    real_nodes = {
        flight.flight_id
        for flight in flight_index.flights_by_icao.get(real_icao, [])
    }

    if not real_nodes:
        raise ValueError(f"ICAO {real_icao!r} not found")

    pia_nodes = {
        flight.flight_id
        for flight in flight_index.get_pia_flights()
    }

    # First remove every other real ICAO.
    allowed_nodes = real_nodes | pia_nodes
    candidate_graph = graph.subgraph(allowed_nodes)

    # Keep only components connected to at least one target-real node.
    connected_nodes = {
        node
        for component in nx.weakly_connected_components(candidate_graph)
        if component & real_nodes
        for node in component
    }

    return candidate_graph.subgraph(connected_nodes).copy()

In [19]:
def get_unique_path_in_dag(
    graph: nx.DiGraph,
    source: str,
    target: str,
) -> list[str] | None:
    if source not in graph or target not in graph:
        return None

    # Only inspect nodes that occur on some source-to-target path.
    relevant_nodes = (
        (nx.descendants(graph, source) | {source})
        & (nx.ancestors(graph, target) | {target})
    )
    relevant_graph = graph.subgraph(relevant_nodes)

    if not nx.is_directed_acyclic_graph(relevant_graph):
        raise ValueError("Expected the flight graph to be a DAG")

    # Count paths, capped at 2 because we only care about uniqueness.
    path_counts: dict[str, int] = {target: 1}

    for node in reversed(list(nx.topological_sort(relevant_graph))):
        if node == target:
            continue

        path_counts[node] = min(
            2,
            sum(
                path_counts.get(successor, 0)
                for successor in relevant_graph.successors(node)
            ),
        )

    if path_counts.get(source, 0) != 1:
        return None

    # Reconstruct the unique path.
    path = [source]
    current = source

    while current != target:
        valid_successors = [
            successor
            for successor in relevant_graph.successors(current)
            if path_counts.get(successor, 0) > 0
        ]

        if len(valid_successors) != 1:
            return None

        current = valid_successors[0]
        path.append(current)

    return path

real_icaos = df_flights.filter(~pl.col("pia")).get_column("icao").unique().to_list()
for real_icao in real_icaos :
    flights = flight_index.flights_by_icao.get(real_icao, [])

    if len(flights) < 2:
        continue

    g = graph_for_real_icao(graph, flight_index, real_icao)

    path = get_unique_path_in_dag(
        g,
        flights[0].flight_id,
        flights[-1].flight_id,
    )

    if path is not None and any(
        flight_index.get_flight_by_id(node).pia
        for node in path
    ):
        print(
            real_icao,
            flights[0].flight_id,
            flights[-1].flight_id,
            path,
        )

a1b6e3 a1b6e3_2026-06-09_22-35 a1b6e3_2026-06-30_23-20 ['a1b6e3_2026-06-09_22-35', 'a0924a_2026-06-14_15-42', 'a0924a_2026-06-14_16-26', 'a1b6e3_2026-06-23_19-23', 'a1b6e3_2026-06-24_16-39', 'a1b6e3_2026-06-24_18-03', 'a1b6e3_2026-06-25_18-26', 'a1b6e3_2026-06-25_19-22', 'a1b6e3_2026-06-27_12-57', 'a1b6e3_2026-06-27_15-44', 'a1b6e3_2026-06-27_16-59', 'a1b6e3_2026-06-27_19-41', 'a0925b_2026-06-29_01-35', 'a1b6e3_2026-06-30_23-20']
ab7db3 ab7db3_2026-06-03_10-56 ab7db3_2026-06-30_14-33 ['ab7db3_2026-06-03_10-56', 'ab7db3_2026-06-03_12-37', 'ab7db3_2026-06-03_19-48', 'ab7db3_2026-06-03_21-16', 'ab7db3_2026-06-07_19-41', 'ab7db3_2026-06-07_22-08', 'ab7db3_2026-06-08_06-53', 'ab7db3_2026-06-08_08-18', 'ab7db3_2026-06-08_21-13', 'ab7db3_2026-06-09_07-01', 'ab7db3_2026-06-10_02-13', 'ab7db3_2026-06-10_03-36', 'a10128_2026-06-15_15-51', 'a10128_2026-06-18_15-53', 'ab7db3_2026-06-30_00-22', 'ab7db3_2026-06-30_13-12', 'ab7db3_2026-06-30_14-33']
ac3275 ac3275_2026-06-01_15-23 ac3275_2026-06-24_20

In [16]:
import polars as pl
df_flights.filter(pl.col("icao") == "a36562")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a36562""","""TWY317""","""N318JS""",2026-06-03 23:16:38.840,"""KSAV""",2026-06-03 23:59:59.120,"""KLVL""",2026-06-03 23:16:38.840,32.118759,-81.200006,-200,null,2026-06-03 23:59:59.120,36.656349,-77.921759,45000,null,false,true,false,false,"""GLF6""","""BANK OF UTAH TRUSTEE""","""GULFSTREAM G650""","""A3"""
"""a36562""","""TWY317""","""N318JS""",2026-06-04 00:00:05.870,"""KRDU""",2026-06-04 00:56:25.890,"""KTEB""",2026-06-04 00:00:05.870,36.667755,-77.909604,45000,null,2026-06-04 00:56:25.890,40.845348,-74.05986,null,null,false,true,false,false,"""GLF6""","""BANK OF UTAH TRUSTEE""","""GULFSTREAM G650""","""A3"""
"""a36562""","""TWY317""","""N318JS""",2026-06-04 22:24:46.540,"""KTEB""",2026-06-04 23:24:51.990,"""CYYZ""",2026-06-04 22:24:46.540,40.853127,-74.05915,275,null,2026-06-04 23:24:51.990,43.670254,-79.610191,null,null,false,true,false,false,"""GLF6""","""BANK OF UTAH TRUSTEE""","""GULFSTREAM G650""","""A3"""
"""a36562""","""TWY317""","""N318JS""",2026-06-05 00:21:13,"""CYYZ""",2026-06-05 07:31:50.570,"""LIML""",2026-06-05 00:21:13,43.677684,-79.599282,475,null,2026-06-05 07:31:50.570,45.429165,9.278825,575,null,false,true,false,false,"""GLF6""","""BANK OF UTAH TRUSTEE""","""GULFSTREAM G650""","""A3"""
"""a36562""","""TWY317""","""N318JS""",2026-06-06 11:19:16.940,"""LIML""",2026-06-06 12:00:48.560,"""LFMN""",2026-06-06 11:19:16.940,45.451091,9.276413,525,null,2026-06-06 12:00:48.560,43.654757,7.203091,null,null,false,true,false,false,"""GLF6""","""BANK OF UTAH TRUSTEE""","""GULFSTREAM G650""","""A3"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a36562""","""TWY317""","""N318JS""",2026-06-24 22:59:54.230,"""LIEO""",2026-06-24 23:38:21.520,"""LFMN""",2026-06-24 22:59:54.230,40.998323,9.661664,7550,null,2026-06-24 23:38:21.520,43.666351,7.218481,null,null,false,true,false,false,"""GLF6""","""BANK OF UTAH TRUSTEE""","""GULFSTREAM G650""","""A3"""
"""a36562""","""TWY317""","""N318JS""",2026-06-25 00:40:14.910,"""LFMN""",2026-06-25 08:34:43.590,"""CYHM""",2026-06-25 00:40:14.910,43.650055,7.207031,-50,null,2026-06-25 08:34:43.590,43.173255,-79.924589,null,null,false,true,false,false,"""GLF6""","""BANK OF UTAH TRUSTEE""","""GULFSTREAM G650""","""A3"""
"""a36562""","""TWY317""","""N318JS""",2026-06-25 09:05:33.640,"""CYHM""",2026-06-25 09:25:34.210,"""CYQA""",2026-06-25 09:05:33.640,43.176407,-79.934543,750,null,2026-06-25 09:25:34.210,44.787048,-79.276036,4325,null,false,true,false,false,"""GLF6""","""BANK OF UTAH TRUSTEE""","""GULFSTREAM G650""","""A3"""
